# Official fast.ai course22 original 03 — Colab adaptation

Original: [03-which-image-models-are-best.ipynb](https://github.com/fastai/course22/blob/230390584ba5b990e20487d68681a9ec5b258eff/clean/03-which-image-models-are-best.ipynb)

Pinned source commit: `230390584ba5b990e20487d68681a9ec5b258eff`. Source SHA256: `cb756a124421579b67df54c4cd98101cdc0625646815fe5b3e238f63af8eade4`.

This additional notebook preserves the full original computational experiments. Existing independently written companions 00–07 remain separate. Set QUICK_RUN=False; run every code cell. Source notebooks and authorship belong to Jeremy Howard and fast.ai. No separate course22 repository license was found at the pinned commit; no fastbook license is applied to these files.

- Historical CSVs deleted from modern timm are restored from the contemporaneous2022-05-13 commit; all rows in the original inference/training benchmark tables and their matching ImageNet catalogue are retained. Current ImageNet table would match just one legacy model and is not used. Original analyses/filters/OLS preserved.

Public publication and competition submission have not been performed. Saved predictions are learning artifacts; the 418 Titanic test targets are unavailable. Original conceptual prompts/headings are retained below.


In [ ]:
# Colab setup. Run this cell before the exercise cells.
from pathlib import Path
import os, sys, zipfile

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  #@param {type:"boolean"}
ARCHIVE_PATH = "/content/drive/MyDrive/HaiLong_Colab_Exercises_2026-10-03.zip"  #@param {type:"string"}

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive', timeout_ms=300000)
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        archive = Path(ARCHIVE_PATH)
        if not archive.is_file():
            print('Select HaiLong_Colab_Exercises_2026-10-03.zip from your computer.')
            uploaded = files.upload()
            choices = [Path(n) for n in uploaded if n.endswith('.zip')]
            if len(choices) != 1:
                raise ValueError('Upload exactly one course ZIP file.')
            archive = choices[0]
        with zipfile.ZipFile(archive) as z:
            base = Path('/content').resolve()
            for member in z.infolist():
                target = (base / member.filename).resolve()
                if not target.is_relative_to(base) or not member.filename.startswith('HaiLong_Colab_Exercises/'):
                    raise ValueError('This is not the expected course archive.')
            z.extractall(base)
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        raise FileNotFoundError('The ZIP must contain HaiLong_Colab_Exercises/colab_runtime.py.')
else:
    # Also permits local checks without assuming the current notebook directory.
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Open this notebook in Colab or run it inside the extracted course folder.')

os.environ['COURSE_SMOKE'] = '1' if QUICK_RUN else '0'
os.environ['WANDB_MODE'] = 'offline'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(COURSE_ROOT))
from colab_runtime import setup_course, finish_run
# Filled individually for each notebook by the builder.

COLAB_CONTEXT = setup_course(COURSE_ROOT, '03_fastai', 'fastai', '20_original_03_image_model_benchmarks.ipynb', USE_GOOGLE_DRIVE, install=IN_COLAB)
if IN_COLAB:
    from packaging.requirements import Requirement
    import importlib.metadata as _metadata, subprocess
    _missing = []
    for _spec in ['plotly>=5,<7', 'statsmodels>=0.14,<1']:
        _requirement = Requirement(_spec)
        try: _version = _metadata.version(_requirement.name)
        except _metadata.PackageNotFoundError: _version = None
        if _version is None or _version not in _requirement.specifier: _missing.append(_spec)
    if _missing: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *_missing])

In [ ]:
# Full-source execution and durable evidence helpers (Colab adaptation).
import hashlib, json, shutil, urllib.request
from datetime import datetime, timezone
if QUICK_RUN or os.environ.get('COURSE_SMOKE') == '1':
    raise ValueError('These original notebooks require full mode: set QUICK_RUN=False.')
RESULTS = COLAB_CONTEXT['folder'] / 'results' / Path(COLAB_CONTEXT['run']['notebook']).stem
MODELS = COLAB_CONTEXT['folder'] / 'models' / Path(COLAB_CONTEXT['run']['notebook']).stem
WORK = COLAB_CONTEXT['folder'] / 'data' / Path(COLAB_CONTEXT['run']['notebook']).stem
for directory in [RESULTS, MODELS, WORK]: directory.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

def record_stage(stage, **evidence):
    target = RESULTS / 'stage_evidence.json'
    started = COLAB_CONTEXT['run']['started_utc']
    data = json.loads(target.read_text(encoding='utf-8')) if target.is_file() else {}
    if data.get('run_started_utc') != started:
        data = {'notebook': COLAB_CONTEXT['run']['notebook'], 'run_started_utc': started, 'stages': {}}
    data['stages'][stage] = {'recorded_utc': datetime.now(timezone.utc).isoformat(), **evidence}
    data.update(quick_run=False, status='observed_stage_outputs', provider_course_completion_claimed=False,
                learner_personal_reflection_complete=False)
    temporary = target.with_suffix('.writing')
    temporary.write_text(json.dumps(data, indent=2, default=str), encoding='utf-8')
    temporary.replace(target)

def verified_download(url, target, sha256):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.is_file():
        temporary = target.with_name(target.name + '.download')
        with urllib.request.urlopen(url, timeout=120) as response, temporary.open('wb') as out:
            shutil.copyfileobj(response, out)
        if hashlib.sha256(temporary.read_bytes()).hexdigest() != sha256:
            raise ValueError('Source download checksum mismatch: ' + url)
        temporary.replace(target)
    if hashlib.sha256(target.read_bytes()).hexdigest() != sha256:
        raise ValueError('Existing source file differs from the pinned original: ' + str(target))
    return target

def save_plot(figure, name):
    if hasattr(figure, 'write_html'):
        figure.write_html(RESULTS / (name + '.html'), include_plotlyjs=True)
    elif hasattr(figure, 'savefig'):
        figure.savefig(RESULTS / (name + '.png'), dpi=150, bbox_inches='tight')
    return figure

record_stage('full_mode_setup', execution_location='hosted_colab' if IN_COLAB else 'local_validation',
             original_exercises_verified_complete=False)

## Evidence required for the original exercises

- Execute every nonempty original code cell top to bottom in hosted Colab; retain outputs and execution records.
- QUICK_RUN=False. Original experiment data coverage, epochs and learning rates are preserved.
- Original source attribution and the per-cell provenance map distinguish original exercises from companions.
- Save full notebook outputs and actual stage/checkpoint/artifact evidence in Drive.
- No provider badge, external publishing, competition submission or personal authorship is certified by execution.
- Analyze pinned original RTX3090/PyTorch1.11 inference and training benchmark tables, all family filtering/scatter/OLS/parameter analyses; these are historical source benchmarks, not measurements of the Colab T4.


## timm

[PyTorch Image Models](https://timm.fast.ai/) (timm) is a wonderful library by Ross Wightman which provides state-of-the-art pre-trained computer vision models. It's like Huggingface Transformers, but for computer vision instead of NLP (and it's not restricted to transformers-based models)!

Ross has been kind enough to help me understand how to best take advantage of this library by identifying the top models. I'm going to share here so of what I've learned from him, plus some additional ideas.

## The data

Ross regularly benchmarks new models as they are added to timm, and puts the results in a CSV in the project's GitHub repo. To analyse the data, we'll first clone the repo:

In [ ]:
# Original published benchmark CSVs, pinned before model identifier changes.
benchmark_path = WORK / 'pytorch-image-models' / 'results'
_benchmark_sources = [{'file': 'benchmark-infer-amp-nhwc-pt111-cu113-rtx3090.csv', 'url': 'https://raw.githubusercontent.com/huggingface/pytorch-image-models/3409bb91568b107a44d82c3552b02ac7edefa032/results/benchmark-infer-amp-nhwc-pt111-cu113-rtx3090.csv', 'sha256': '6b5b489bf7d35a0b37e10aea0d8b4cca03b9f6efc5b33ff3153914e41f8d89a8'}, {'file': 'benchmark-train-amp-nhwc-pt111-cu113-rtx3090.csv', 'url': 'https://raw.githubusercontent.com/huggingface/pytorch-image-models/3409bb91568b107a44d82c3552b02ac7edefa032/results/benchmark-train-amp-nhwc-pt111-cu113-rtx3090.csv', 'sha256': '5f2736e337d56bd61ad5d45baa8fdb895ae63056bcb8e3a0713c5b264d16e78c'}, {'file': 'results-imagenet.csv', 'url': 'https://raw.githubusercontent.com/huggingface/pytorch-image-models/3409bb91568b107a44d82c3552b02ac7edefa032/results/results-imagenet.csv', 'sha256': '8515c0b07f1135c721beb4a0e64f9813f8a00f172246b4a2da42ed600e1c5fa2'}]
for _source in _benchmark_sources:
    verified_download(_source['url'], benchmark_path / _source['file'], _source['sha256'])
os.chdir(benchmark_path)
record_stage('original_benchmark_inputs', sources=_benchmark_sources, benchmark_hardware='RTX3090', measured_in_this_Colab=False)

In [ ]:
import pandas as pd
df_results = pd.read_csv('results-imagenet.csv')

In [ ]:
def get_data(part, col):
    df = pd.read_csv(f'benchmark-{part}-amp-nhwc-pt111-cu113-rtx3090.csv').merge(df_results, on='model')
    df['secs'] = 1. / df[col]
    df['family'] = df.model.str.extract(r'^([a-z]+?(?:v2)?)(?:\d|_|$)')
    df = df[~df.model.str.endswith('gn')]
    df.loc[df.model.str.contains('in22'),'family'] = df.loc[df.model.str.contains('in22'),'family'] + '_in22'
    df.loc[df.model.str.contains('resnet.*d'),'family'] = df.loc[df.model.str.contains('resnet.*d'),'family'] + 'd'
    return df[df.family.str.contains('^re[sg]netd?|beit|convnext|levit|efficient|vit|vgg')]

In [ ]:
df = get_data('infer', 'infer_samples_per_sec')

assert len(df) > 100, 'Historical model IDs did not join correctly.'
df.to_csv(RESULTS / 'inference_analysis.csv', index=False)

## Inference results

In [ ]:
import plotly.express as px
w,h = 1000,800

def show_all(df, title, size):
    return px.scatter(df, width=w, height=h, size=df[size]**2, title=title,
        x='secs',  y='top1', log_x=True, color='family', hover_name='model', hover_data=[size])

In [ ]:
save_plot(show_all(df, 'Inference', 'infer_img_size'), 'inference_all')

In [ ]:
subs = 'levit|resnetd?|regnetx|vgg|convnext.*|efficientnetv2|beit'

In [ ]:
def show_subs(df, title, size):
    df_subs = df[df.family.str.fullmatch(subs)]
    return px.scatter(df_subs, width=w, height=h, size=df_subs[size]**2, title=title,
        trendline="ols", trendline_options={'log_x':True},
        x='secs',  y='top1', log_x=True, color='family', hover_name='model', hover_data=[size])

In [ ]:
save_plot(show_subs(df, 'Inference', 'infer_img_size'), 'inference_selected')

In [ ]:
save_plot(px.scatter(df, width=w, height=h,
    x='param_count_x',  y='secs', log_x=True, log_y=True, color='infer_img_size',
    hover_name='model', hover_data=['infer_samples_per_sec', 'family']
), 'parameters_and_latency')

## Training results

In [ ]:
tdf = get_data('train', 'train_samples_per_sec')

assert len(tdf) > 100, 'Historical model IDs did not join correctly.'
tdf.to_csv(RESULTS / 'training_analysis.csv', index=False)

In [ ]:
save_plot(show_all(tdf, 'Training', 'train_img_size'), 'training_all')

In [ ]:
save_plot(show_subs(tdf, 'Training', 'train_img_size'), 'training_selected')

record_stage('all_original_benchmark_analyses', inference_models=len(df), training_models=len(tdf), figures=['inference_all','inference_selected','parameters_and_latency','training_all','training_selected'], includes_ols=True)

In [ ]:
record_stage('reached_final_cell', all_previous_cells_independently_verified=False)
finish_run(COLAB_CONTEXT)